# KAT-VAD v2 — P6 / **D5 explanation on T2-test** (Amendment 7, H1–H4)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§16 Amendment 7**, written after batch 1 and before this number.
Batch 1's D5 **FAILED on T2-val** (micro Δ −0.1089, macro Δ −0.0073) — the phase-4 references trained on T2-val's
windows. Here all four checkpoints are scored on **T2-test**, unseen by every one of them. **No training. No DoTA.**

| step | what | GPU? |
|---|---|---|
| 0 | setup | — |
| 1 | preflight; stage the parent T2 dir + plain CLIP cache | no |
| 2 | `core.evaluate` × 4 (A0 s2099, phase-4 s2024 / s2025 / s2026), exactly as phase 4 called it | optional (CPU ≈ minutes) |
| 3 | **H1** code identity · **H2** out-of-sample D5 · **H3** verdict · H4 prints → `d5_t2test.md` | — |

A1 is **not** scored on T2-test (H4). Everything is written under `Thesis-V2/outputs/v2_pilot/d5_t2test/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/, v1 outputs (phase-4 references)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # write: kat-vad/ (code), outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'
V1_OUTPUTS = Path(DRIVE) / 'outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 16. Amendment 7' in addendum, 'this upload predates Amendment 7 -- re-upload branch v2'

DATASET  = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
SEED     = 2099
MARGIN   = constants.V2_D5_MARGIN                       # 0.02 (H2)
H1_TOL   = 1e-3                                         # H1: |Δ| vs phase 4's own T2-test numbers
T2_TEST_WINDOWS = 1106

T2       = constants.DATA_ROOT / DATASET                # parent dir: frame_labels_test.json = T2-test windows
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET
PILOT    = constants.OUTPUT_ROOT / 'v2_pilot'
OUT      = PILOT / 'd5_t2test'
CKPTS = {
    'A0_s2099': PILOT / 'A0' / f's{SEED}' / 'checkpoint_last.pt',
    'p4_s2024': V1_OUTPUTS / 'DADA2000_orig_phase4' / 's2024' / 'stage2_kip_off' / 'checkpoint_last.pt',
    'p4_s2025': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2025' / 'stage2_kip_off' / 'checkpoint_last.pt',
    'p4_s2026': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2026' / 'stage2_kip_off' / 'checkpoint_last.pt',
}
REFS = ('p4_s2024', 'p4_s2025', 'p4_s2026')
# Phase 4's own T2-test results (REPORTS/DADA2000_orig_phase4/results_s*_kip_off_t2.json). s2024 is H1's target;
# s2025/s2026 belong to the lost mid-training checkpoints and are printed only (H4).
PHASE4_T2 = {'p4_s2024': (0.6230302243992729, 0.633409862583937),
             'p4_s2025': (0.6163772247274282, 0.6351581481718938),
             'p4_s2026': (0.6152462593975954, 0.6056965986262889)}
P4_REPORT = V1_OUTPUTS / 'REPORTS' / 'DADA2000_orig_phase4' / 'results_s2024_kip_off_t2.json'
LOCAL = Path('/content/d5')
LOCAL.mkdir(parents=True, exist_ok=True)
for name, path in CKPTS.items():
    print(f'{name:9s}', path)


In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU -- fine, eval only'


In [ ]:
import hashlib
import json
import shutil
import subprocess
import time

import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst):
    """Copy a Drive dir/file to VM disk once (bit-identical; C2 untouched)."""
    if dst.exists():
        return dst
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive and fsync every file (C10)."""
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight + stage

In [ ]:
for path in (T2 / constants.FRAME_LABELS_TEST_FILENAME, CLIP_DIR, *CKPTS.values()):
    assert path.exists(), f'missing {path}'

# J10 on every checkpoint: its own global_step == the last step of the metrics.jsonl beside it.
for name, ckpt in CKPTS.items():
    step, last = ckpt_step(ckpt), metrics_last(ckpt.parent / 'metrics.jsonl')
    assert last is not None and step == last['global_step'], f'{name}: J10 FAILS (ckpt {step}, metrics {last})'
    print(f'{name}: step {step} (J10 ok)')

if P4_REPORT.exists():                                  # the hardcoded H1 target must be the file's
    rep = json.loads(P4_REPORT.read_text())
    assert (rep['auc'], rep['auc_macro']) == PHASE4_T2['p4_s2024'], 'phase-4 report != the H1 target above'
    print('H1 target matches', P4_REPORT.name)
else:
    print(f'NOTE: {P4_REPORT} not on Drive -- using the numbers recorded in the repo')

L_T2 = stage(T2, LOCAL / 'stage' / 'corpus')
L_CLIP = stage(CLIP_DIR, LOCAL / 'stage' / 'clip')
test = json.loads((L_T2 / constants.FRAME_LABELS_TEST_FILENAME).read_text())
assert len(test) == T2_TEST_WINDOWS, f'{len(test)} T2-test windows, expected {T2_TEST_WINDOWS}'
print(f'T2-test: {len(test)} windows (parent dir)')


## 2. Score on T2-test — `core.evaluate`, phase 4's exact call

No `--set model.*` / `kip.*` (C34): the architecture comes from each checkpoint. Re-run the cell after a disconnect;
finished results on Drive are reused.


In [ ]:
def evaluate(name, ckpt):
    drive_dir = OUT / name
    if (drive_dir / 'results.json').exists():
        print(f'{name}: reusing {drive_dir / "results.json"}')
        return json.loads((drive_dir / 'results.json').read_text())
    local = LOCAL / 'eval' / name
    run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt,
         '--set', f'data.dataset={DATASET}',
         '--data-dir', L_T2, '--clip-dir', L_CLIP, '--output-dir', local],
        log=LOCAL / f'{name}_eval.log')
    publish(local / 'results.json', drive_dir / 'results.json')
    return json.loads((local / 'results.json').read_text())


RES = {name: evaluate(name, ckpt) for name, ckpt in CKPTS.items()}
for name, r in RES.items():
    assert r['num_videos'] == T2_TEST_WINDOWS, f"{name}: scored {r['num_videos']} windows"
    print(f"{name}: micro {r['auc']:.4f} (norm {r['score_norm']}) | macro {r['auc_macro']:.4f}")


## 3. H1 · H2 · H3 · H4 → `d5_t2test.md` — then **stop**

In [ ]:
import numpy as np

val = {name: json.loads(p.read_text())['guardrails']['micro'] for name, p in {
    'A0_s2099': PILOT / 'A0' / f's{SEED}' / 'diag' / 'diag.json',
    **{r: PILOT / 'd5_refs' / r.split('_')[1] / 'diag.json' for r in REFS}}.items()}

micro = {n: RES[n]['auc'] for n in RES}
macro = {n: RES[n]['auc_macro'] for n in RES}

h1_d = (micro['p4_s2024'] - PHASE4_T2['p4_s2024'][0], macro['p4_s2024'] - PHASE4_T2['p4_s2024'][1])
H1 = all(abs(d) <= H1_TOL for d in h1_d)

ref_micro, ref_macro = np.mean([micro[r] for r in REFS]), np.mean([macro[r] for r in REFS])
h2_d = (micro['A0_s2099'] - ref_micro, macro['A0_s2099'] - ref_macro)
H2 = all(abs(d) <= MARGIN for d in h2_d)

if H1 and H2:
    verdict = 'D5: FAIL on T2-val, EXPLAINED -- in-sample bias of the references. D5 no longer blocks P7.'
elif not H1:
    verdict = 'H1 FAILS -- the v2 tree changed v1\'s scoring path. STOP, debug; P7 blocked.'
else:
    verdict = 'H1 passes, H2 FAILS -- A0\'s training differs (v2 dataset dir / KNN). STOP, debug; P7 blocked.'

rows = []
for n in RES:
    p4 = PHASE4_T2.get(n)
    rows.append(f"| {n} | {micro[n]:.4f} | {macro[n]:.4f} | "
                f"{'—' if p4 is None else f'{p4[0]:.4f} / {p4[1]:.4f}'} | {val[n]:.4f} | {val[n] - micro[n]:+.4f} |")
text = '\n'.join([
    f'# v2 pilot — D5 on T2-test (Amendment 7)', '',
    '| checkpoint | T2-test micro | macro | phase-4 printed | T2-val micro | T2-val − T2-test |',
    '|---|---|---|---|---|---|', *rows, '',
    f"**H1** (s2024 reproduces phase 4, tol {H1_TOL}): Δ micro {h1_d[0]:+.5f}, Δ macro {h1_d[1]:+.5f} → "
    f"{'PASS' if H1 else 'FAIL'}",
    f"**H2** (A0 vs ref mean on T2-test, margin {MARGIN}): ref mean {ref_micro:.4f} / {ref_macro:.4f}; "
    f"Δ micro {h2_d[0]:+.4f}, Δ macro {h2_d[1]:+.4f} → {'PASS' if H2 else 'FAIL'}",
    f'**H3** {verdict}', '',
    'H4 (printed): s2025/s2026 phase-4 numbers belong to the lost snapshots; the last column is the in-sample',
    'inflation (references) vs A0 (T2-val unseen).', ''])
(LOCAL / 'd5_t2test.md').write_text(text)
(LOCAL / 'd5_t2test.json').write_text(json.dumps({
    'addendum': 'core/docs/v2/PREREG_ADDENDUM.md §16 (H1-H4)', 't2_test': {n: [micro[n], macro[n]] for n in RES},
    't2_val_micro': val, 'h1': {'delta': h1_d, 'tol': H1_TOL, 'pass': H1},
    'h2': {'ref_mean': [ref_micro, ref_macro], 'delta': h2_d, 'margin': MARGIN, 'pass': H2},
    'verdict': verdict}, indent=2))
publish(LOCAL / 'd5_t2test.md', OUT / 'd5_t2test.md')
publish(LOCAL / 'd5_t2test.json', OUT / 'd5_t2test.json')
print(text)


Paste `v2_pilot/d5_t2test/d5_t2test.md` back to Claude. Whatever H3 says, A0, its recipe and batch 1's T2-val
read-out are not changed (O6, H3).